# Lección 6.6 — Por qué los ensambles de árboles dominan los datos tabulares

> **Módulo 6 · Ensambles: la sabiduría de la multitud** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 60 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lecciones 6.2–6.4 (Random Forest y boosting), Lección 5.3 (KNN), Lección 1.6 ("no hay almuerzo gratis")

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué hace especiales a los **datos tabulares** (hojas de cálculo, bases de datos) frente a imágenes o texto.
- Enumerar y **demostrar** tres razones técnicas por las que los árboles brillan ahí: funciones **irregulares**, columnas **inútiles** y el **significado de cada columna**.
- Reconocer cuándo **no** conviene un ensamble de árboles (extrapolar, relaciones suaves con pocos datos, imágenes y texto crudos).
- Aplicar una **receta** práctica para empezar cualquier proyecto con datos tabulares.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 En esta lección usaremos una **red neuronal** sencilla (`MLPClassifier` de scikit-learn) como rival de los árboles. No necesitas saber cómo funciona por dentro (lo verás en el Módulo 9): por ahora, piensa en ella como un modelo muy flexible que dibuja fronteras **suaves**.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression, RidgeCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_classification, load_breast_cancer, load_wine, load_digits, load_diabetes, fetch_openml
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")      # silencia avisos de convergencia de la red neuronal
plt.rcParams["figure.figsize"] = (8, 4.5)

def red_neuronal():
    return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=1_000,
                                                         early_stopping=True, random_state=0))
print("✅ Todo listo")

---
## 1. 📖 La intuición: el sastre y la plastilina

En la última década, las redes neuronales revolucionaron las **imágenes**, el **audio** y el **texto** (Módulos 9–13). Pero si miras las competencias de Kaggle con **datos tabulares** (una tabla con filas y columnas: clientes, transacciones, pacientes…), los ganadores casi siempre usan **XGBoost, LightGBM, CatBoost o Random Forest**. Un estudio famoso (Grinsztajn, Oyallon y Varoquaux, NeurIPS 2022) lo confirmó en 45 conjuntos de datos: *"los modelos basados en árboles siguen superando al aprendizaje profundo en datos tabulares típicos"*.

Una analogía:
- Un **árbol** es como un **sastre**: corta la tela con tijeras, **columna por columna**, en línea recta: "si edad > 40, de un lado; si no, del otro". Puede hacer cortes bruscos donde haga falta.
- Una **red neuronal** es como **plastilina**: se adapta de forma suave y continua a cualquier forma, pero le cuesta hacer esquinas afiladas, y mezcla todas las columnas desde el principio.

En una imagen, los píxeles vecinos se parecen y lo que importa son formas suaves: gana la plastilina. En una tabla, cada columna **significa algo distinto** (edad, saldo, ciudad) y abundan los **umbrales** ("mayores de 65", "más de 3 atrasos"): gana el sastre.

### ✋ Ejercicio 1 — ¿Tabla o no tabla?
Para cada caso, di si son datos **tabulares** y si apostarías de entrada por un ensamble de árboles:
1. Historial de 50,000 clientes de un banco: edad, ingreso, número de productos, atrasos, ciudad.
2. 100,000 fotos de radiografías para detectar neumonía.
3. Mediciones de 300 sensores de una fábrica, una fila por hora, para predecir fallas.
4. Reseñas de restaurantes en texto libre para saber si son positivas.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Tabular** → ensamble de árboles, sin duda como primera opción.
2. **No tabular** (imágenes): redes convolucionales o Transformers (Módulo 10). Un árbol vería cada píxel como una columna sin relación con sus vecinos.
3. **Tabular** (aunque con orden en el tiempo: cuidado con la validación, Lección 4.6). Árboles, con variables como "promedio de las últimas 24 horas".
4. **No tabular** (texto): modelos de lenguaje (Módulos 11–13). Aunque si conviertes el texto en columnas (p. ej. *embeddings*, Lección 11.2), un boosting sobre ellas puede funcionar bien.
</details>

---
## 2. 📖 Razón 1: los datos tabulares están llenos de "esquinas"

En el mundo real abundan las reglas con **umbrales**: tarifas que cambian por tramos, edades de jubilación, límites de crédito, días festivos. La función que hay que aprender tiene **saltos** y **esquinas**, no curvas suaves.

Vamos a llevarlo al extremo con un **tablero de ajedrez** de 4×4: la clase cambia bruscamente al cruzar cada línea. (Un 5 % de las etiquetas tiene ruido.)

In [ ]:
rng = np.random.default_rng(0)
n = 2_000
X_tab = rng.uniform(0, 4, (n, 2))
y_tab = ((np.floor(X_tab[:, 0]) + np.floor(X_tab[:, 1])) % 2 == 0).astype(int)   # casillas blancas y negras
y_tab = np.where(rng.random(n) < 0.05, 1 - y_tab, y_tab)                           # 5 % de ruido

plt.figure(figsize=(4.5, 4.5))
plt.scatter(*X_tab.T, c=y_tab, cmap="coolwarm", s=6); plt.title("El tablero: una función llena de esquinas"); plt.show()

### 🔮 Ejercicio 2 — Sastre contra plastilina
Compararemos con validación cruzada una **red neuronal** (dos capas de 64 neuronas), un **Random Forest** y un **HistGradientBoosting**. ¿Quién ganará y por cuánto?

✍️ *Mi apuesta:*

In [ ]:
rivales = {
    "Red neuronal":   red_neuronal(),
    "Random Forest":  RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "HistGradBoost":  HistGradientBoostingClassifier(random_state=0),
}
for nombre, m in rivales.items():
    print(f"{nombre:<14}: exactitud = {cross_val_score(m, X_tab, y_tab, cv=5).mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Los árboles ganan por goleada: Random Forest ≈**0.936** y HistGradientBoosting ≈**0.932**, contra ≈**0.744** de la red neuronal. Como hay un 5 % de ruido, lo máximo razonable es ≈0.95: los árboles están muy cerca del techo. A la red le cuesta dibujar 16 casillas con esquinas exactas a partir de curvas suaves; con mucho más entrenamiento y ajuste mejoraría, pero los árboles lo logran **de fábrica**.
</details>

### 🔮 Ejercicio 3 — ¿Y si la frontera es suave?
Ahora la regla es una **diagonal** suave: la clase es 1 si `x₀ + x₁ > 4` (más algo de ruido). Sin esquinas. ¿Quién gana ahora?

✍️ *Mi apuesta:*

In [ ]:
y_diag = (X_tab[:, 0] + X_tab[:, 1] + rng.normal(0, 0.5, n) > 4).astype(int)
for nombre, m in rivales.items():
    print(f"{nombre:<14}: exactitud = {cross_val_score(m, X_tab, y_diag, cv=5).mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Ahora gana la **red neuronal**: ≈**0.909** contra ≈0.898 de los dos ensambles. Una frontera diagonal y suave es justo lo que la "plastilina" dibuja bien, mientras que los árboles tienen que aproximarla con una **escalera** de cortes horizontales y verticales.

Moraleja: **no hay almuerzo gratis** (Lección 1.6). Los árboles no son mejores "en general": son mejores para el **tipo de función** que suele aparecer en las tablas.
</details>

---
## 3. 📖 Razón 2: las tablas tienen muchas columnas inútiles

En una tabla real es normal tener decenas de columnas que **no sirven** para predecir (el día de la semana en que se capturó el dato, el número de sucursal, códigos internos…). ¿Qué pasa con cada modelo cuando las agregamos?

- Un **árbol** simplemente **no pregunta** por las columnas que no ayudan: las ignora casi gratis.
- **KNN** mide distancias con **todas** las columnas: el ruido "aleja" a los vecinos verdaderos (la maldición de la dimensionalidad, Lección 2.8).
- Una **red neuronal** mezcla todas las columnas desde la primera capa y tiene que **aprender** a ignorar las inútiles.

### 🔮 Ejercicio 4 — Agrega ruido
Partimos de 2,000 ejemplos con **6 columnas útiles** y vamos agregando 20 y luego 100 columnas de **puro ruido**. Ordena de más a menos afectado: red neuronal, KNN, Random Forest, HistGradientBoosting.

✍️ *Mi orden:*

In [ ]:
X_util, y_util = make_classification(n_samples=2_000, n_features=6, n_informative=6, n_redundant=0, random_state=0)
modelos = {
    "Red neuronal":  red_neuronal(),
    "KNN (k=15)":    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
    "HistGradBoost": HistGradientBoostingClassifier(random_state=0),
}
tabla = {}
for n_ruido in [0, 20, 100]:       # ✏️
    X_con_ruido = np.hstack([X_util, rng.normal(size=(len(X_util), n_ruido))])
    tabla[f"+{n_ruido} inútiles"] = {nombre: cross_val_score(m, X_con_ruido, y_util, cv=5).mean() for nombre, m in modelos.items()}
pd.DataFrame(tabla).round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

| | +0 | +20 | +100 | Caída |
|---|---|---|---|---|
| KNN | 0.942 | 0.817 | 0.702 | **−0.24** |
| Red neuronal | 0.928 | 0.878 | 0.833 | −0.10 |
| Random Forest | 0.936 | 0.916 | 0.874 | −0.06 |
| HistGradBoost | 0.944 | 0.930 | 0.919 | **−0.03** |

Orden de más a menos afectado: **KNN → red neuronal → Random Forest → HistGradientBoosting**. Sin ruido, los cuatro empatan prácticamente; con 100 columnas inútiles, el boosting casi no se inmuta, porque cada árbol solo usa las columnas que reducen el error. El bosque sufre un poco más porque, con `max_features="sqrt"`, en muchas divisiones **solo** le tocan columnas de ruido para elegir (Lección 6.2).
</details>

---
## 4. 📖 Razón 3: cada columna significa algo

Los árboles hacen preguntas sobre **una columna a la vez**. Eso es una suposición: que las columnas, **tal como vienen**, tienen sentido por separado ("edad", "saldo"). En una tabla suele ser verdad. En una imagen, no: un píxel suelto no significa nada.

¿Qué pasa si **mezclamos** las columnas? Tomemos el tablero y **girémoslo 45°**: es exactamente el mismo problema, pero ahora cada columna nueva es una **mezcla** de las dos originales, y las esquinas del tablero ya no están alineadas con los ejes.

### 🔮 Ejercicio 5 — Gira el tablero
1. ¿A quién le afectará más el giro: a los árboles o a la red neuronal?
2. ¿Por qué?

✍️ *Mis predicciones:*

In [ ]:
angulo = np.pi / 4                       # ✏️ prueba otros ángulos (np.pi / 8, np.pi / 2)
rotacion = np.array([[np.cos(angulo), -np.sin(angulo)], [np.sin(angulo), np.cos(angulo)]])
X_girado = X_tab @ rotacion

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(*X_tab.T, c=y_tab, cmap="coolwarm", s=4); axes[0].set_title("Original")
axes[1].scatter(*X_girado.T, c=y_tab, cmap="coolwarm", s=4); axes[1].set_title("Girado 45°")
plt.tight_layout(); plt.show()

for nombre, m in rivales.items():
    original = cross_val_score(m, X_tab, y_tab, cv=5).mean()
    girado = cross_val_score(m, X_girado, y_tab, cv=5).mean()
    print(f"{nombre:<14}: original={original:.3f}  girado={girado:.3f}  cambio={girado - original:+.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. A los **árboles**: Random Forest baja de ≈0.935 a ≈0.893 y HistGradientBoosting de ≈0.932 a ≈0.889 (unos **4 puntos**). La red neuronal casi no cambia (≈0.744 → ≈0.730).
2. Porque los árboles cortan **en paralelo a los ejes**. Con el tablero alineado, cada borde de casilla es un solo corte; girado, cada borde se vuelve una diagonal que hay que aproximar con escalones. A la red le da casi igual la orientación: no hace preguntas sobre columnas sueltas.

(Aun girado, los árboles siguen ganándole a la red en el tablero: las esquinas siguen ahí. Pero su ventaja se redujo.)

Esta es una de las tres razones del estudio de 2022: las redes neuronales "no ven" la orientación de los datos (les da igual girarlos), y en las tablas **la orientación importa**, porque cada columna tiene un significado propio. Los árboles aprovechan esa estructura.
</details>

---
## 5. 📖 Las razones prácticas (las que más se notan en el día a día)

Más allá de la teoría, los ensambles de árboles ganan porque **dan menos trabajo**:

| Ventaja práctica | Por qué | Lección |
|---|---|---|
| **No hay que escalar** | Las preguntas "¿x > umbral?" no cambian si x está en pesos o en miles de pesos | 5.4 |
| **Mezclan tipos sin problema** | Numéricas, binarias y categóricas (nativas en boosting moderno) | 6.4 |
| **Toleran faltantes** | HistGB, XGBoost, LightGBM, CatBoost los manejan solos | 6.4 |
| **Robustos a atípicos en las X** | Un ingreso de 10 millones solo cae "del lado derecho" de un corte | 5.4 |
| **Capturan interacciones solos** | "Joven **y** sin contrato **y** muchas llamadas" sale de preguntas encadenadas | 5.4, 6.3 |
| **Buenos valores por defecto** | Un Random Forest o un HistGB de fábrica ya suele ser competitivo | 6.2, 6.4 |
| **Rápidos de entrenar** | Minutos (o segundos) en una laptop, sin GPU | 6.4 |
| **Explicables a nivel global** | Importancias por permutación, dependencias parciales | 6.2, 7.6 |

### ✋ Ejercicio 6 — El costo de preparar
Una analista tiene una tabla con: `edad` (años), `ingreso` (pesos, con algunos valores de 50 millones), `ciudad` (300 valores), `antiguedad` (con 12 % de faltantes) y `tiene_tarjeta` (sí/no). Haz una lista de **qué preparación** necesitaría para usar **KNN** o una **red neuronal**, y cuál para un **HistGradientBoosting**.

✍️ *Mi lista:*

<details><summary><b>👀 Ver respuesta</b></summary>

**KNN o red neuronal:**
- Rellenar los faltantes de `antiguedad` (y quizá agregar un indicador de faltante).
- Codificar `ciudad` (one-hot con 300 columnas, o target encoding con cuidado de no tener fuga).
- Codificar `tiene_tarjeta` como 0/1.
- **Escalar** todo, y tratar los ingresos extremos (quizás con logaritmo), porque distorsionan distancias y gradientes.

**HistGradientBoosting:** convertir `ciudad` y `tiene_tarjeta` al tipo `category` de pandas y usar `categorical_features="from_dtype"`. Y ya. Menos pasos = menos lugares donde equivocarse (y menos riesgo de fuga, Lección 2.6).
</details>

---
## 6. 📖 Cuándo NO usar un ensamble de árboles

Los árboles tienen límites claros. El más importante: **no extrapolan** (Lección 5.4). Predicen siempre algo parecido a lo que vieron.

### 🔮 Ejercicio 7 — Las ventas que crecen
Una tienda vende cada mes un poco más que el anterior (tendencia creciente + ruido). Entrenamos con los meses **1 a 36** y predecimos los meses **37 a 48**. ¿Qué hará el boosting con el futuro? ¿Y una regresión lineal?

✍️ *Mi predicción:*

In [ ]:
meses = np.arange(1, 49)
ventas = 100 + 5 * meses + rng.normal(0, 10, len(meses))     # +5 unidades por mes, con ruido
pasado, futuro = meses <= 36, meses > 36

boosting = HistGradientBoostingRegressor(random_state=0).fit(meses[pasado, None], ventas[pasado])
lineal = LinearRegression().fit(meses[pasado, None], ventas[pasado])

plt.plot(meses, ventas, "ko", ms=3, label="ventas reales")
plt.plot(meses, boosting.predict(meses[:, None]), "r", label="boosting")
plt.plot(meses, lineal.predict(meses[:, None]), "b--", label="regresión lineal")
plt.axvline(36.5, color="gray", ls=":"); plt.text(37, ventas.min(), "futuro →")
plt.legend(); plt.xlabel("Mes"); plt.ylabel("Ventas"); plt.show()

for nombre, m in [("Boosting", boosting), ("Regresión lineal", lineal)]:
    error = np.mean(np.abs(m.predict(meses[futuro, None]) - ventas[futuro]))
    print(f"{nombre:<17} error medio en el futuro (MAE): {error:5.1f} unidades")

<details><summary><b>👀 Ver respuesta</b></summary>

El boosting predice el futuro como una línea **plana**, al nivel de los últimos meses que vio: su error medio en el futuro es de ≈**118** unidades, contra ≈**8.5** de la regresión lineal, que simplemente continúa la tendencia de +5 por mes. Para el boosting, "mes 45" cae en la misma hoja que "mes 36": nunca vio nada más allá.

Arreglo típico si quieres seguir usando árboles: predecir el **cambio** respecto al mes anterior (o quitar la tendencia antes) en lugar del valor absoluto, y dejar que la tendencia la maneje un modelo lineal.
</details>

### 📖 Resumen: ¿cuándo pensarlo dos veces?

| Situación | Por qué los árboles sufren | Alternativa |
|---|---|---|
| Hay que **extrapolar** (tendencias, valores fuera del rango visto) | Predicen "lo último que vieron" | Modelo lineal, o árboles sobre el **cambio** |
| Relación **suave y casi lineal**, con **pocos datos** | El árbol aproxima una recta con escalones y necesita muchos datos | Regresión lineal/logística regularizada (ganó en la diabetes, Lección 6.5) |
| **Imágenes, audio, texto** crudos | Cada píxel/palabra suelta no significa nada por sí solo | Redes neuronales (Módulos 9–13) |
| Necesitas una **fórmula** o reglas completas y legibles | Cientos de árboles no se leen | Árbol simple, modelo lineal, o explicaciones (Lección 7.6) |
| **Muy pocos datos** (decenas de filas) | Poca información para cortar bien | Modelos simples + conocimiento del dominio |

---
## 7. 📖 La receta para datos tabulares

```
1. Línea base tonta       → "predecir el promedio / la clase mayoritaria" (Lección 4.7)
2. Línea base simple      → regresión lineal/logística (¿ya es suficiente? ¿se puede explicar?)
3. Random Forest de fábrica → casi nunca sale mal; te dice cuánto hay por ganar
4. Boosting moderno       → HistGB / LightGBM / XGBoost / CatBoost con parada temprana
5. Ajuste razonable       → learning_rate, tamaño de árbol, mínimo por hoja (Módulo 7)
6. Mejores datos          → nuevas variables (Lección 2.4): suelen ganar más que el paso 5
7. ¿Ensamble de ensambles? → stacking solo si la mejora vale la complejidad (Lección 6.5)
```

### ✋ Ejercicio 8 — Aplica la receta
Un compañero dice: *"Para predecir la rotación de empleados (5,000 filas, 25 columnas) empecé directamente con un stacking de XGBoost, LightGBM y una red neuronal, con 200 combinaciones de hiperparámetros. Me dio AUC = 0.81."* ¿Qué le preguntarías?

✍️ *Mis preguntas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- ¿Cuánto da una **regresión logística**? ¿Y un **Random Forest de fábrica**? Si dan 0.80, la complejidad no está justificada.
- ¿Cómo evaluaste? ¿Validación cruzada sin tocar la prueba? Con 200 combinaciones, el riesgo de "sobreajustar a la validación" es real (Lección 7.3).
- ¿La parada temprana usó datos de validación, no de prueba (Lección 6.4)?
- ¿Probaste **mejores variables** (antigüedad en el puesto, cambios de jefe, aumentos recientes)? Suelen pesar más que el modelo.
- ¿Quién va a mantener y explicar un stacking de tres modelos?
</details>

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| "Las redes neuronales son lo más moderno, las usaré en mi tabla" | Moda, no evidencia | Comparar contra un boosting en validación cruzada |
| El boosting predice valores planos para casos nuevos fuera de rango | No extrapolan | Revisar el rango de las variables; modelo lineal o predecir cambios |
| Un modelo lineal le gana al boosting | Relación casi lineal y pocos datos | ¡Usa el lineal! Es más simple y explicable |
| Diste vueltas un mes ajustando hiperparámetros por 0.002 de AUC | Rendimientos decrecientes | Volver a los datos: nuevas variables, limpieza, más ejemplos |
| Pasaste píxeles crudos a un Random Forest y se estancó | Datos no tabulares | Redes convolucionales (Módulo 10) o características extraídas |

---
## 9. 📋 Ficha: ensambles de árboles para datos tabulares

| | Ensambles de árboles (Random Forest, boosting) en tablas |
|---|---|
| **Por qué ganan** | Funciones con esquinas · Ignoran columnas inútiles · Respetan el significado de cada columna · Poca preparación |
| **Primera opción** | Random Forest de fábrica (robusto) → boosting moderno con parada temprana (más preciso) |
| **✅ Fortalezas** | Precisión de punta en tablas, sin escalar, con faltantes y categorías, rápidos y con buenos valores por defecto |
| **❌ Debilidades** | No extrapolan · Fronteras en escalera para relaciones suaves · No sirven para datos crudos no tabulares |
| **Úsalos cuando…** | Tienes una tabla de cientos a millones de filas con columnas con significado |
| **Evítalos cuando…** | Necesitas extrapolar, tienes poquísimos datos con una relación simple, o trabajas con imágenes/audio/texto crudos |

## 10. 🧠 Resumen en 5 puntos
1. En **datos tabulares**, los ensambles de árboles siguen siendo el estado del arte; en imágenes, audio y texto, ganan las redes neuronales.
2. **Razón 1:** las tablas están llenas de **umbrales y esquinas**, que los árboles cortan con facilidad.
3. **Razón 2:** los árboles **ignoran** las columnas inútiles; KNN y las redes sufren con ellas.
4. **Razón 3:** los árboles aprovechan que **cada columna significa algo**; si mezclas (giras) las columnas, pierden su ventaja.
5. No son mágicos: **no extrapolan** y un modelo lineal puede ganarles con relaciones suaves y pocos datos. **Mide siempre** contra líneas base simples.

---
## 11. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈35 min).

### 11.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál de estos conjuntos es **tabular**?
- a) Grabaciones de llamadas · b) Una tabla de pólizas con edad, prima, tipo de auto y siniestros · c) Fotos de autos chocados

**2.** Agregas 50 columnas de ruido. ¿Qué modelo esperas que sufra **más**?
- a) HistGradientBoosting · b) Random Forest · c) KNN

**3.** Giras (mezclas) las columnas de una tabla. ¿Qué modelo pierde más?
- a) La red neuronal · b) El ensamble de árboles · c) Ninguno

**4.** Entrenas un boosting con precios de casas de hasta $5 M y preguntas por una de $20 M. ¿Qué esperas?
- a) Una predicción de ≈$20 M · b) Una predicción parecida a las casas más caras que vio · c) Un error

**5.** ¿Cuál es el **segundo** paso de la receta?
- a) Stacking · b) Un modelo lineal/logístico simple · c) Buscar 500 combinaciones de hiperparámetros

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)**
2. **c)** KNN mide distancias con todas las columnas (Ejercicio 4).
3. **b)** Los árboles dependen de que cada columna tenga sentido por separado (Ejercicio 5).
4. **b)** No extrapolan (Ejercicio 7).
5. **b)** Después de la línea base tonta, un modelo simple: a veces ya es suficiente.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** Si giras una foto 45°, sigue siendo el mismo gato. Si "giras" las columnas de una tabla de clientes (cada columna nueva = 0.7 × edad + 0.7 × ingreso…), ¿sigue teniendo sentido? Usa esta idea para explicar con tus palabras por qué las redes neuronales funcionan bien en imágenes y los árboles en tablas.

**B. Caso de análisis 💼** Una aseguradora quiere predecir el costo de los siniestros. Su equipo de ciencia de datos propone un **LightGBM**. El de actuaría prefiere un **modelo lineal generalizado** (GLM, de la familia de la regresión lineal) porque *"siempre se ha hecho así"*. ¿Qué argumentos tiene cada lado? ¿Cómo resolverías el desacuerdo con datos?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** En una foto, lo que importa son **formas** que se ven igual desde cualquier ángulo: no hay un "eje" especial, y por eso un modelo al que le da igual la orientación (la red) no pierde nada. En una tabla, cada eje **es** una cantidad con significado ("edad", "ingreso"): una mezcla 0.7 × edad + 0.7 × ingreso no significa nada para el negocio. Los árboles hacen preguntas sobre ejes con significado, y por eso aprovechan la estructura de las tablas; las redes la ignoran y tienen que redescubrirla.

**B.**
- **LightGBM:** suele ser más preciso, captura umbrales e interacciones solo, maneja faltantes y categorías.
- **GLM:** es explicable coeficiente por coeficiente, los reguladores lo conocen, se comporta de forma predecible fuera del rango (extrapola con una tendencia) y es fácil de auditar.
- **Cómo resolverlo:** comparar los dos con la **misma** validación (idealmente temporal: entrenar con años anteriores y evaluar en el más reciente), con una métrica de negocio (error en dinero). Si LightGBM gana por poco, el GLM puede ser la mejor opción; si gana por mucho, se puede usar LightGBM para **descubrir** umbrales e interacciones y luego incorporarlos al GLM, o usar LightGBM con explicaciones (Lección 7.6).
</details>

### 11.3 🧩 Práctica integradora: el gran torneo

Pondremos a prueba todo lo dicho con **seis** conjuntos de datos y **cinco** modelos. Usaremos validación cruzada de 5 pliegues (exactitud en clasificación, R² en regresión).

| Datos | Filas | Columnas | Tipo | Qué es |
|---|---|---|---|---|
| `cancer` | 569 | 30 | clasificación | Medidas de células en una biopsia (Lección 5.4) |
| `vinos` | 178 | 13 | clasificación (3 clases) | Análisis químico de vinos de tres productores |
| `digitos` | 1,797 | 64 | clasificación (10 clases) | **Imágenes** de 8×8 píxeles de dígitos escritos a mano |
| `diabetes` | 442 | 10 | regresión | Avance de la diabetes (Lección 6.5) |
| `tablero` | 2,000 | 2 | clasificación | El tablero de esta lección |
| `censo` | 10,000 | 13 | clasificación | Una muestra del censo de la Lección 6.4 (¿gana más de 50K?): tabla real con categorías y faltantes |

Los cuatro primeros vienen incluidos en scikit-learn; son conjuntos "de libro": **pequeños y limpios**. Para el censo, todos los modelos reciben la misma preparación (rellenar faltantes, one-hot y escalar), así la comparación es justa.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 🔮 — Apuesta antes de correr.** Para cada conjunto, ¿quién crees que ganará: un modelo **lineal**, **KNN**, la **red neuronal**, **Random Forest** o **HistGradientBoosting**?

✍️ *Mis apuestas:* cancer → ___ · vinos → ___ · digitos → ___ · diabetes → ___ · tablero → ___ · censo → ___

**Paso 2 ▶️ — El torneo** (tarda unos minutos).

In [ ]:
datos = {
    "cancer":   load_breast_cancer(return_X_y=True) + ("clasif",),
    "vinos":    load_wine(return_X_y=True) + ("clasif",),
    "digitos":  load_digits(return_X_y=True) + ("clasif",),
    "diabetes": load_diabetes(return_X_y=True) + ("regres",),
    "tablero":  (X_tab, y_tab, "clasif"),
}
censo = fetch_openml("adult", version=2, as_frame=True).frame.sample(10_000, random_state=0)
datos["censo"] = (censo.drop(columns=["class", "fnlwgt"]), (censo["class"] == ">50K").astype(int).values, "clasif")

def preparar_censo(modelo):      # misma preparación para todos: rellenar + one-hot + escalar
    prep = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
         make_column_selector(dtype_include="category")),
        (StandardScaler(), make_column_selector(dtype_include="number")))
    return make_pipeline(prep, modelo)

def concursantes(tipo):
    escalar = StandardScaler()
    if tipo == "clasif":
        return {"Lineal": make_pipeline(escalar, LogisticRegression(max_iter=2_000)),
                "KNN": make_pipeline(escalar, KNeighborsClassifier(n_neighbors=10)),
                "Red neuronal": red_neuronal(),
                "Random Forest": RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
                "HistGradBoost": HistGradientBoostingClassifier(random_state=0)}
    return {"Lineal": RidgeCV(alphas=np.logspace(-3, 3, 20)),
            "KNN": make_pipeline(escalar, KNeighborsRegressor(n_neighbors=10)),
            "Red neuronal": make_pipeline(escalar, MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=2_000,
                                                                early_stopping=True, random_state=0)),
            "Random Forest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
            "HistGradBoost": HistGradientBoostingRegressor(random_state=0)}

torneo = pd.DataFrame({nombre: {modelo: cross_val_score(preparar_censo(m) if nombre == "censo" else m, X, y, cv=5).mean()
                                for modelo, m in concursantes(tipo).items()}
                       for nombre, (X, y, tipo) in datos.items()}).round(3)
torneo.loc["🏆 Ganador"] = torneo.idxmax()
torneo

<details><summary><b>👀 Ver respuesta</b></summary>

| | cancer | vinos | digitos | diabetes | tablero | censo |
|---|---|---|---|---|---|---|
| 🏆 | Lineal (0.981) | Lineal (0.983) | KNN / RF (0.938) | Lineal (0.478) | HistGB (0.932) | HistGB (≈0.866) |

¡Sorpresa! En los cuatro conjuntos "de libro", los ensambles de árboles **no** ganan claramente: el modelo **lineal** gana en cancer, vinos y diabetes, y en dígitos KNN empata con Random Forest. Los árboles ganan donde hay **esquinas** (tablero) y en la única **tabla real y desordenada** (censo, con categorías y faltantes), donde HistGradientBoosting (≈0.866) supera por más de 2 puntos a la logística (≈0.844), a la red neuronal y al Random Forest (≈0.840 los dos).

Esto **no** contradice la lección: la afirmación es sobre datos tabulares **típicos** del mundo real (miles de filas, columnas heterogéneas, umbrales, ruido). Conjuntos pequeños y limpios, con relaciones casi lineales, son justo el terreno donde un modelo simple brilla. Por eso la receta empieza con una **línea base lineal**.
</details>

**Paso 3 ✋ — Explica las "derrotas".** Para cada conjunto donde **no** ganó un ensamble de árboles, explica por qué usando las razones de esta lección.

✍️ *Mis explicaciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **cancer** y **vinos:** pocos ejemplos (569 y 178) y clases que se separan casi con una recta en las medidas escaladas. Una logística no necesita más; los árboles aproximan esa recta con escalones.
- **diabetes:** relación casi **lineal** y solo 442 pacientes (la misma conclusión que en la Lección 6.5).
- **digitos:** son **imágenes**: cada columna es un píxel, sin significado por sí solo. KNN, que compara imágenes enteras "píxel a píxel", funciona muy bien. (Con imágenes más grandes, las redes convolucionales del Módulo 10 ganarían a todos.)
</details>

**Paso 4 ✏️ — Dale una oportunidad a los árboles en los dígitos.** Las imágenes de 8×8 son tan pequeñas que casi parecen una tabla. Cambia el número de árboles o prueba `max_features` y mira cuánto se acercan los árboles a los ganadores.

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
bosque_dig = RandomForestClassifier(n_estimators=500, max_features="sqrt", random_state=0, n_jobs=-1)   # ✏️
print(f"Random Forest en dígitos: {cross_val_score(bosque_dig, X_dig, y_dig, cv=5).mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con 500 árboles el bosque llega a ≈**0.938**, igual que con 300: ya estaba en su techo (más árboles no ayudan, Lección 6.2). Empata con KNN, así que en imágenes tan pequeñas los árboles se defienden bien, pero no tienen ninguna ventaja especial. Con fotos reales de miles de píxeles, la distancia con las redes convolucionales sería enorme.
</details>

**Paso 5 ✋ — Tu regla.** Escribe en dos frases tu propia regla para elegir modelo en un problema nuevo, basada en lo que viste en el torneo.

<details><summary><b>👀 Ver respuesta</b></summary>

Una regla razonable: *"Si los datos son una tabla con columnas con significado, empiezo con una línea base lineal y un ensamble de árboles (Random Forest y luego boosting), y me quedo con el más simple que no pierda de forma clara. Si son imágenes, audio o texto crudos, o si la relación es casi lineal con muy pocos datos o necesito extrapolar, pruebo primero otros modelos."*

**Siempre** con validación cruzada y una línea base: el torneo mostró que hasta los mejores pierden en el terreno equivocado.
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Grinsztajn, Oyallon y Varoquaux (2022), *Why do tree-based models still outperform deep learning on typical tabular data?* (NeurIPS). Las tres razones de esta lección vienen de ahí.
- 📄 **Artículo:** Shwartz-Ziv y Armon (2021), *Tabular Data: Deep Learning is Not All You Need*.
- 📘 **Lectura práctica:** el "Kaggle Grandmasters Playbook" y las soluciones ganadoras publicadas en Kaggle para competencias tabulares.

---
⬅️ **Anterior:** [6.5 Stacking y voting: un comité de modelos distintos](6.5-stacking-y-voting.ipynb) · ➡️ **Siguiente:** [6.7 Proyecto del módulo: El tasador automático 🏠](6.7-proyecto-del-modulo.md)